# Evaluating a RAG App with Ragas + LangChain: Annotated Walkthrough

> **Annotated copy.** The code is identical to `raga_evaluation_live_session.ipynb`. Only comments and markdown cells were added.

## The short version

This notebook does two things:

1. **Builds a small RAG question-answering app.** It answers questions about *Ragas* using only four short documents.
2. **Grades that app automatically with Ragas.** A second LLM acts as a **judge**. It scores every answer on **correctness** (pass/fail) and **relevance** (1 to 5) and explains each score.

You end up with a pass rate, an average relevance score, and a CSV file that holds every question, answer, score, and the judge's reasoning.

## Why evaluate at all?

LLM output is free-form text and can change from one run to the next, so a normal unit test like `assert answer == "..."` doesn't work. You still need a **repeatable, measurable** way to answer questions like these:

* *"I changed the prompt / model / `TOP_K`. Did the app get better or worse?"*
* *"When an answer is wrong, did **retrieval** fail (the right document wasn't found) or did **generation** fail (the LLM misused a good document)?"*

An **evaluation harness** answers both: a fixed test set, fixed metrics, and saved results you can compare across runs.

## Key terms

| Term | Plain-English meaning |
|---|---|
| **RAG** (Retrieval-Augmented Generation) | Look up relevant text first, then ask the LLM to answer *using that text*. This reduces made-up answers (hallucinations). |
| **Embedding** | A list of numbers that represents the meaning of a text. Texts with similar meaning get similar vectors. |
| **Vector store** | A database of embeddings that can find "the most similar texts to this question". |
| **Retriever / `TOP_K`** | The component that returns the *k* most similar documents for a question. |
| **LCEL chain** (`a \| b \| c`) | LangChain's way of piping one step's output into the next step. |
| **Golden dataset** | Questions paired with human-written *expected answers*, i.e. the ground truth. |
| **LLM-as-a-judge** | Using an LLM, guided by a written rubric, to grade another LLM's output. |
| **Rubric / Likert scale** | A written scoring guide. A Likert scale is an ordered 1-5 rating where each number has a defined meaning. |
| **Reference-based vs. reference-free metric** | Reference-based metrics compare against an expected answer (correctness here). Reference-free metrics need only the question and the response (relevance here). |
| **Experiment** | Run the app and the metrics over every row of the dataset, then store the results. |

## How the pieces fit together

```
PART A: the app under test (RAG)                PART B: the evaluation harness (Ragas)
-----------------------------------             -----------------------------------------
4 Documents --embed--> InMemoryVectorStore      Golden Dataset (question, expected_answer)
                              |                              |
question --> Retriever (top-k docs)                          v  for each row (concurrently)
                              |                    @experiment run_rag_evaluation(row)
                     format_documents()                      |
                              |                     1. rag_query(question)  <---- calls Part A
               prompt | llm | StrOutputParser                |
                              |                     2. correctness_metric (judge: pass/fail)
                           answer  ----------->     3. relevance_metric   (judge: 1-5)
                                                             |
                                                  results -> pandas -> summary + CSV
```

## Map of the cells

| Step | What happens |
|---|---|
| 0-1 | Kernel sanity check, plus a commented-out catalog of Ragas' built-in metrics (reference only). |
| 1-2 | Load the OpenAI key from `.env` and set the config knobs (generator model, embedding model, judge model, `TOP_K`). |
| 3-4 | Create the LangChain LLM and embedding objects, then make one smoke-test call. |
| 5-7 | Build the knowledge base, embed it into a vector store, and create a retriever. |
| 8-10 | Prompt, then LCEL chain, then `rag_query()`: the complete RAG app. |
| 11-14 | Ragas imports, the golden dataset, and the judge LLM client. |
| 15-16 | Two custom metrics: **correctness** (pass/fail) and **relevance** (1-5). |
| 17-19 | The `@experiment` per-row function, `main()` (summary and CSV), and the cell that runs it all. |
| End | **What the results tell us**: an analysis of the saved run. |

## Before you run it

* A `.env` file with `OPENAI_API_KEY=...` in this folder or a parent folder.
* Packages: `ragas`, `langchain-openai`, `langchain-core`, `openai`, `python-dotenv`, `pandas`, `nest_asyncio`.
* `JUDGE_MODEL` must be a model your OpenAI account can access.
* **Cost:** each question uses about 1 embedding call, 1 generator call, and 2 judge calls. With 4 questions that's very cheap, but it grows linearly with dataset size.
* **Outputs written:** `evals/datasets/ragas_langchain_dataset.csv`, `evals/experiments/ragas_langchain_experiment.csv` (saved by Ragas), and `evals/experiments/ragas_langchain_results.csv` (saved by `main()`).

## Design notes for experienced readers

* **The judge is separate from the generator** (`JUDGE_MODEL` vs. `MODEL`). This reduces self-preference bias, where a model grades its own style favourably.
* **The metrics are custom** (`DiscreteMetric`, `NumericMetric`) rather than Ragas' built-ins such as `Faithfulness` or `ContextPrecision`. They're easy to read and adapt. Because `rag_query()` already returns `retrieved_contexts`, adding built-in retrieval and faithfulness metrics is a small step.
* **Retrieval is logged but not scored.** `retrieved_document_ids` is saved for debugging. One easy extension: add an `expected_doc_id` column and compute a retrieval hit rate.
* **`temperature=0` is not full determinism.** For decisions that matter, run the experiment more than once and compare the averages.


In [17]:
# STEP 0: Sanity check.
# If this prints, the Jupyter kernel is running Python and you're ready to go.
# Nothing RAG-specific happens here.
print("Hello, World!")

Hello, World!


In [ ]:
# REFERENCE ONLY: everything in this cell is commented out and never runs.
# This is a catalog of the metrics that ship with Ragas, grouped by what they measure.
# This notebook does NOT use them. It builds two custom metrics instead
# (correctness_metric and relevance_metric, further down). Keep this list as a menu for later.
#
# [Group 1] RAG metrics
#   Retrieval quality:
#     ContextPrecision     - are the retrieved chunks relevant and ranked near the top?
#     ContextRecall        - did we retrieve everything needed to answer?
#     ContextEntityRecall  - are the key entities from the reference present in the context?
#     NoiseSensitivity     - how often does irrelevant retrieved text lead to wrong claims?
#   Answer quality:
#     AnswerRelevancy      - does the answer address the question?
#     Faithfulness         - is every claim supported by the retrieved context (no hallucination)?
#     FactualCorrectness   - do the claims match a reference answer?
#     SemanticSimilarity   - is the answer's meaning close to the reference (embedding-based)?
# [Group 2] Multimodal versions of faithfulness/relevance (text + images).
# [Group 3] Agent / tool-use metrics:
#   did the agent stay on topic, call the right tools with the right arguments,
#   and reach the user's goal (with or without a reference outcome)?
# [Group 4] Comparison with a reference, including traditional NLP metrics.
#   BLEU / chrF / ROUGE / ExactMatch / StringPresence / NonLLMStringSimilarity don't need an LLM,
#   so they're cheap and deterministic. But they only reward word overlap:
#   a correct answer phrased differently can score low.
# from ragas.metrics.collections import (
#     ContextPrecision,
#     ContextRecall,
#     ContextEntityRecall,
#     NoiseSensitivity,
#     AnswerRelevancy,
#     Faithfulness,
#     FactualCorrectness,
#     SemanticSimilarity,
# )
# from ragas.metrics.collections import (
#     MultiModalFaithfulness,
#     MultiModalRelevance,
# )
# from ragas.metrics.collections import (
#     TopicAdherence,
#     ToolCallAccuracy,
#     ToolCallF1,
#     AgentGoalAccuracyWithReference,
#     AgentGoalAccuracyWithoutReference,
# )
# from ragas.metrics.collections import (
#     FactualCorrectness,
#     SemanticSimilarity,
#     NonLLMStringSimilarity,
#     DistanceMeasure,
#     BleuScore,
#     CHRFScore,
#     RougeScore,
#     StringPresence,
#     ExactMatch,
# )

# [Group 5] Build-your-own metrics.
#   Decorators that turn a plain Python function into a metric that returns a
#   label (discrete), a number (numeric) or an ordering (ranking).
# from ragas.metrics import (
#     discrete_metric,
#     numeric_metric,
#     ranking_metric,
# )
# Rubric-based LLM metrics:
#   DomainSpecificRubrics   - one scoring rubric shared by every row
#   InstanceSpecificRubrics - each row carries its own rubric
# from ragas.metrics.collections import (
#     DomainSpecificRubrics,
#     InstanceSpecificRubrics,
# )

                             RAG
                              │
          ┌───────────────────┼───────────────────┐
          │                   │                   │
   OpenAIEmbeddings     Vector Store         Retriever
                                                  │
                                                  ▼
                                          Retrieved Docs
                                                  │
                                                  ▼
                                      ChatPromptTemplate
                                                  │
                                                  ▼
                                           ChatOpenAI
                                                  │
                                                  ▼
                                        StrOutputParser
                                                  │
                                                  ▼
                                             Response
                                                  │
                                                  │
                         RAGAS EVALUATION          │
                              │                   │
                              ▼                   ▼
                         Dataset              Response
                              │                   │
                              └─────────┬─────────┘
                                        ▼
                                   @experiment
                                        │
                         ┌──────────────┴─────────────┐
                         │                            │
                 DiscreteMetric                NumericMetric
                  Correctness                   Relevance
                         │                            │
                         └──────────────┬─────────────┘
                                        ▼
                                  Score + Reason
                                        │
                                        ▼
                                Experiment Results
                                        │
                                        ▼
                                      CSV

In [ ]:
# Instructor's notes: two terms used by the metrics further down.
#   Rubric: a written scoring guide that spells out what each score means
#           (e.g. "PASS if ... / FAIL if ..."). Rubrics make LLM-judge scores
#           more consistent and easier to audit.
#   Likert: an ordered rating scale, usually 1-5, where every number has a
#           defined meaning (1 = completely irrelevant ... 5 = completely relevant).
#           relevance_metric below is a Likert-style rubric.
# This cell is all comments, so running it does nothing.
# Rubric
# Likert

In [18]:
# STEP 1: Load secrets.
# Never hardcode API keys in a notebook: they end up in git history.
# Put them in a .env file (OPENAI_API_KEY=sk-...) that is listed in .gitignore.
import os
from dotenv import load_dotenv

# Reads .env (looking in this folder, then parent folders) and copies each
# KEY=value into the process's environment variables.
# By default it does NOT overwrite variables that are already set.
load_dotenv()

# os.getenv returns None if the variable is missing (it doesn't crash).
# Note: ChatOpenAI, OpenAIEmbeddings and AsyncOpenAI all read OPENAI_API_KEY
# from the environment automatically. We read it here only to validate it.
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Fail fast: a clear error here beats a confusing authentication error
# ten cells later.
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env")

print("API key loaded successfully")

API key loaded successfully


In [19]:
# STEP 2: Configuration knobs.
# Keeping them in one place makes experiments easy: change one value, re-run,
# and compare scores.
#
# Generator: the LLM that writes the answers for our RAG app.
# This is the "system under test".
MODEL = "gpt-4o"

# Embedding model: turns text into vectors (lists of numbers) so we can find
# documents whose meaning is close to the question.
# The SAME model must embed both the documents and the questions.
EMBEDDING_MODEL = "text-embedding-3-small"

# Judge: a separate LLM that grades the generator's answers (LLM-as-a-judge).
# Using a different, ideally stronger, model than the generator reduces
# self-preference bias, where a model rates its own style favourably.
# It must be a model name your OpenAI account can access.
JUDGE_MODEL = "gpt-5.6-terra"

# How many documents the retriever returns per question.
#   Too small -> the needed fact may be missing (hurts recall).
#   Too large -> more noise and more tokens (can confuse the LLM, costs more).
# With only 4 documents, k=2 sends half the knowledge base every time.
TOP_K = 2

In [20]:
# STEP 3: LangChain building blocks.
# ChatOpenAI       - wrapper around OpenAI chat models (the generator)
# OpenAIEmbeddings - wrapper around OpenAI embedding models
from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings,
)

# Document = a standard container for text:
#   page_content (the text itself) + metadata (a dict of extras such as an ID or source).
from langchain_core.documents import Document

# ChatPromptTemplate = a prompt with {placeholders} that are filled in at run time.
from langchain_core.prompts import (
    ChatPromptTemplate,
)

# StrOutputParser = converts the LLM's AIMessage object into a plain Python string.
from langchain_core.output_parsers import (
    StrOutputParser,
)

# InMemoryVectorStore = a simple vector database that lives in RAM.
# Good for demos and tests, but the data disappears when the kernel restarts.
# Production systems use a persistent store (FAISS, Chroma, pgvector, Pinecone, ...).
from langchain_core.vectorstores import (
    InMemoryVectorStore,
)

In [21]:
# STEP 4: Create the generator LLM and the embedding model.
# No API call happens yet. These are configured client objects.
llm = ChatOpenAI(
    model=MODEL,
    # temperature=0 makes the model choose its most likely words, so answers
    # are as repeatable as possible. That matters for evaluation: score changes should
    # come from YOUR changes, not from randomness. (OpenAI models are still not 100%
    # deterministic, so expect small run-to-run differences.)
    temperature=0,
)

# Used by the vector store to embed the documents now and each question later.
embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
)

In [22]:
# Smoke test: one real API call to confirm that the key, model name and network work.
# .invoke() returns an AIMessage. The text is in .content and token usage is in
# .usage_metadata (see the output below). It costs a few tokens.
llm.invoke("hi how are you?")

AIMessage(content="Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 12, 'total_tokens': 41, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4c910a9acf', 'id': 'chatcmpl-EkBt3p6zvMSiiRP4I09mvvALaLIt8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1db1d-3ad0-7d81-acdf-4f5e87eb25eb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'output_tokens': 29, 'total_tokens': 41, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning

In [23]:
# ============================================================
# 6. KNOWLEDGE BASE
# ============================================================

# STEP 5: The knowledge base, i.e. the ONLY information the RAG app may answer from.
# Four short Documents about Ragas and RAG.
#
#   page_content     -> the text that gets embedded and searched
#   metadata.doc_id  -> a stable, human-readable ID. We log it later to see WHICH
#                       documents were retrieved for each question, which is how we
#                       tell retrieval problems apart from generation problems.
#
# .strip() removes the leading and trailing newlines of the triple-quoted text.
# In a real app, documents come from loaders (PDFs, web pages, ...) and are split
# into smaller chunks first. Here each document is already one small chunk.
DOCUMENTS = [

    Document(
        page_content="""
Ragas is an open-source evaluation framework designed
for evaluating and improving LLM applications.

It supports RAG evaluation, agent evaluation,
custom metrics, datasets, experiments,
and synthetic test data generation.
""".strip(),
        metadata={
            "doc_id": "ragas_intro",
        },
    ),

    Document(
        page_content="""
Ragas experiments allow developers to systematically
evaluate AI applications, track evaluation results,
compare different versions, and store experiment outputs.
""".strip(),
        metadata={
            "doc_id": "ragas_experiments",
        },
    ),

    Document(
        page_content="""
Ragas metrics measure different properties of AI systems.

Metrics can evaluate factual correctness,
faithfulness, answer relevancy,
retrieval quality, agent behavior,
and application-specific requirements.
""".strip(),
        metadata={
            "doc_id": "ragas_metrics",
        },
    ),

    Document(
        page_content="""
RAG stands for Retrieval-Augmented Generation.

A RAG system retrieves relevant information
from a knowledge base and provides that information
to an LLM for answer generation.
""".strip(),
        metadata={
            "doc_id": "rag_intro",
        },
    ),

]

# Quick check that all 4 documents were created.
print(
    "\nKnowledge Base Documents:",
    len(DOCUMENTS),
)


Knowledge Base Documents: 4


In [24]:
# STEP 6: Build the vector store (the "index").
# It receives the embedding model so it can embed text by itself.
vector_store = InMemoryVectorStore(embedding=embeddings,)

# Embeds every document (one call to the embeddings API) and stores the vector
# plus the document in memory. It returns auto-generated internal IDs (the UUIDs in
# the output), which are different from our own metadata["doc_id"].
# WARNING: re-running this cell adds the same documents AGAIN (duplicates), so top-k
# can return the same text twice. Re-run the cell above first to get a fresh store.
vector_store.add_documents(DOCUMENTS)

['2fc6240c-8e36-45d8-877f-268861ae2e94',
 'c5a89a74-cde6-4259-b166-4105392e02fa',
 '6756b9de-1a6a-43c7-ab46-bb05d015c8d7',
 '4606ace8-4b87-4ad4-8039-6166fb3d1098']

In [25]:
# STEP 7: Turn the vector store into a Retriever. It's an object with
# .invoke(question) / .ainvoke(question) that returns the most similar Documents.
# Under the hood: embed the question -> compare it (cosine similarity) with every
# stored vector -> return the best k. search_kwargs={"k": TOP_K} sets k.
retriever = vector_store.as_retriever(search_kwargs={"k": TOP_K,})

In [26]:
# STEP 8: The generation prompt. {context} and {question} are filled in at run time.
#
# Design choices that matter for evaluation:
#   * "ONLY use the supplied context" + "say I don't know" -> fewer hallucinations;
#     answers stay grounded in (faithful to) the retrieved documents.
#   * Trade-off: a very strict prompt can make the model refuse even when the
#     context does contain the answer, and "keep it concise" can drop details the
#     expected answer needs. BOTH happen in this notebook's results (see the
#     "What the results tell us" section at the bottom).
# Evaluation is how you measure trade-offs like these instead of guessing.
prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer the user's question ONLY using
the supplied context.

Rules:

1. Use only the provided context.
2. Do not invent information.
3. Do not use outside knowledge.
4. If the answer is not present in the context,
   say:

   "I don't know based on the provided context."

5. Keep the answer concise and factual.


CONTEXT:

{context}


QUESTION:

{question}


ANSWER:
"""
)

In [27]:
# Helper: merge the retrieved Documents into ONE string for the {context}
# placeholder. A blank line ("\n\n") separates the chunks so the LLM can tell them apart.
# Input: list[Document] -> Output: str
def format_documents(documents,):
    return "\n\n".join(document.page_content for document in documents)

In [28]:
# STEP 9: The generation chain, written in LCEL (LangChain Expression Language).
# "|" pipes each step's output into the next one, like a Unix pipe:
#
#   {"context": ..., "question": ...}
#       -> prompt            (fills the template -> chat messages)
#       -> llm               (calls the generator -> AIMessage)
#       -> StrOutputParser() (AIMessage -> plain string)
#
# Retrieval is deliberately NOT part of this chain. rag_query() below calls the
# retriever separately so it can keep the retrieved documents for evaluation.
rag_chain = prompt | llm | StrOutputParser()

In [29]:
# ============================================================
# 12. COMPLETE LANGCHAIN RAG APPLICATION
# ============================================================

# STEP 10: The complete RAG app as one async function.
# question in -> answer + evidence out. This is the "system under test" that Ragas evaluates.
#
# Why async? The Ragas experiment runs several questions at the same time. With async,
# those network calls overlap instead of waiting for each other.
#
# Why return more than the answer? The retrieved contexts and doc IDs let us tell
# two kinds of failure apart:
#   retrieval failure  -> the right document was never found
#   generation failure -> the right document was found, but the LLM answered badly
async def rag_query( question: str):
    
    # --------------------------------------------------------
    # STEP 1
    # Retrieve relevant documents using LangChain Retriever
    # --------------------------------------------------------

    # ainvoke = async version of invoke. Returns up to TOP_K Documents, most similar first.
    relevant_documents = await retriever.ainvoke(question)
    
    # Just the raw texts. This is the format Ragas' context-based metrics
    # (e.g. Faithfulness, ContextPrecision) expect, if you add them later.
    retrieved_contexts = [ document.page_content for document in relevant_documents]
    
    # .get() returns None instead of raising KeyError if a document has no doc_id.
    retrieved_document_ids = [document.metadata.get("doc_id") for document in relevant_documents]

    # --------------------------------------------------------
    # STEP 2
    # Create context
    # --------------------------------------------------------

    context = format_documents(relevant_documents)

    # --------------------------------------------------------
    # STEP 3
    # Generate answer using LangChain LCEL chain
    # --------------------------------------------------------

    # The dict keys must match the {placeholders} in the prompt template exactly.
    answer = await rag_chain.ainvoke(
        {
            "context": context,
            "question": question,
        }
    )

    # Return a dict so the caller (the experiment) can use whichever parts it needs.
    return {

        "answer":answer,

        "retrieved_contexts": retrieved_contexts,

        "retrieved_document_ids": retrieved_document_ids

    }

In [30]:
# asyncio runs async functions (coroutines). We need it to drive the evaluation.
import asyncio
# Path = file paths that work on any OS; used to create the output folders.
from pathlib import Path
# pandas = tables (DataFrames); used to summarise the scores and save them to CSV.
import pandas as pd

In [31]:
# STEP 11: Ragas building blocks.
#   Dataset        - a table of test cases (rows) stored through a backend (here: CSV files)
#   experiment     - decorator that turns a per-row async function into an experiment
#                    you can run over a whole Dataset. It runs rows concurrently, shows
#                    a progress bar and saves the results.
#   llm_factory    - wraps an LLM client so Ragas metrics can use it as the judge
#   DiscreteMetric - the judge picks ONE label from a fixed list (e.g. pass / fail)
#   NumericMetric  - the judge returns a number inside a range (e.g. 1-5)
#
# The "TqdmWarning: IProgress not found" in the output is harmless. It only means
# ipywidgets isn't installed, so a plain-text progress bar is used instead.
from ragas import (
    Dataset,
    experiment,
)

from ragas.llms import (
    llm_factory,
)

from ragas.metrics import (
    DiscreteMetric,
    NumericMetric,
)

/Users/manny/Documents/git_repos/Full-stack-GenAI/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [32]:
# STEP 12: The golden (evaluation) dataset: questions plus the answers a
# human considers correct. The judge compares the app's response with
# expected_answer to decide "correctness".
#
# Tips for real projects:
#   * aim for dozens to hundreds of rows, covering easy, hard and edge cases
#   * include questions whose answer is NOT in the knowledge base, to check that
#     the app says "I don't know" instead of hallucinating
def load_dataset():
    # Make sure the output folder exists (exist_ok=True: no error if it already does).
    Path("evals").mkdir(exist_ok=True)

    # An empty Ragas Dataset. With backend "local/csv" and root_dir "evals",
    # it is stored at: evals/datasets/ragas_langchain_dataset.csv
    dataset = Dataset(
        name="ragas_langchain_dataset",
        backend="local/csv",
        root_dir="evals",
    )

    # golden dataset(ground truth)
    # Each row is a dict, and its keys become CSV columns. They must match what
    # run_rag_evaluation() reads: row["question"] and row["expected_answer"].
    data_samples = [

        {
            "question": "What is Ragas?",

            "expected_answer": """
Ragas is an evaluation framework
for LLM applications.
""".strip(),
        },

        {
            "question": "How do Ragas experiments work?",

            "expected_answer": """
Ragas experiments provide systematic evaluation,
track results, compare different application versions,
and store experiment outputs.
""".strip(),
        },

        {
            "question": "What do Ragas metrics do?",

            "expected_answer": """
Ragas metrics evaluate properties and quality
of AI or LLM application outputs.
""".strip(),
        },

        {
            "question": "What is RAG?",

            "expected_answer": """
RAG stands for Retrieval-Augmented Generation.

It retrieves relevant information from a knowledge
source and provides that context to an LLM
before answer generation.
""".strip(),
        },

    ]

    # Add every row to the dataset (in memory)...
    for sample in data_samples:
        dataset.append(sample)

    # ...then write it to the CSV file. A saved test set can be versioned,
    # shared, reviewed and reused across many experiment runs.
    dataset.save()

    return dataset

In [33]:
# The official OpenAI SDK's async client.
# Ragas' llm_factory expects a raw OpenAI-SDK client, not LangChain's ChatOpenAI,
# so we create a separate client just for the judge. It's async because the
# metrics are scored with `await`.
from openai import AsyncOpenAI

In [34]:
# STEP 13: HTTP client for the judge LLM.
ragas_openai_client = AsyncOpenAI(
    # os.environ[...] raises KeyError if the key is missing (already checked in Step 1).
    api_key=os.environ[ "OPENAI_API_KEY"],
    # Give up on any single request after 60 s instead of hanging forever.
    timeout=60.0,
    # Automatically retry temporary errors (rate limits, 5xx) up to 3 times with
    # back-off. This helps when many rows run in parallel and hit rate limits.
    max_retries=3,
)

In [35]:
# STEP 14: Wrap the client + JUDGE_MODEL as a Ragas LLM.
# Metrics use it to ask the judge for a STRUCTURED reply with two parts:
#   value  -> the label or score
#   reason -> the judge's explanation
# We pass evaluator_llm explicitly to every metric's .ascore(...) call below.
evaluator_llm = llm_factory(
    JUDGE_MODEL,
    client=ragas_openai_client,
)

In [36]:
# STEP 15: Metric #1, CORRECTNESS (pass / fail). This is a REFERENCE-BASED metric.
# It asks: "Does the response contain the key facts of the expected answer,
# without contradicting it?"
#
# How it works: at scoring time Ragas fills {question}, {expected_answer} and
# {response} with the keyword arguments passed to .ascore(...) (so the names must
# match exactly), sends the prompt to the judge, and accepts only a value from
# allowed_values.
#
# Why pass/fail instead of 1-10? Binary decisions are easier for an LLM (and for
# humans) to make consistently, and they add up to an easy-to-read pass rate.
# The explicit PASS/FAIL rules form a rubric, which makes decisions consistent and
# the judge's reasons easy to audit.
correctness_metric = DiscreteMetric(

    name="correctness",

    prompt="""
Evaluate whether the model response correctly answers
the user's question according to the expected answer.

QUESTION:

{question}


EXPECTED ANSWER:

{expected_answer}


MODEL RESPONSE:

{response}


Evaluation Rules:

PASS if:

1. The response contains the important information
   from the expected answer.

2. The response is factually correct.

3. The response answers the user's question.


FAIL if:

1. Important information is missing.

2. The response contradicts the expected answer.

3. The response does not answer the question.


Return only:

pass

or

fail
""",

    # A LIST = the only labels the judge may return.
    allowed_values=[
        "pass",
        "fail",
    ],

)

In [37]:
# STEP 16: Metric #2, RELEVANCE (score 1-5). This is a REFERENCE-FREE metric.
# It asks: "Does the response actually address what was asked?"
# Notice it does NOT get expected_answer: it judges whether the answer is on topic,
# not whether it is complete or correct. An answer can be fully relevant yet
# incomplete, which is exactly why we use two metrics (see "What is RAG?" in the
# results).
# The prompt is a Likert-scale rubric: each number has a written meaning.
relevance_metric = NumericMetric(

    name="relevance",

    prompt="""
Evaluate how relevant the model response is
to the user's question.


QUESTION:

{question}


MODEL RESPONSE:

{response}


Use this rubric:

1 = Completely irrelevant

2 = Mostly irrelevant

3 = Partially relevant

4 = Mostly relevant

5 = Completely relevant


Return only the numerical score.
""",

    # For NumericMetric, a TUPLE (min, max) is a RANGE, not a list of choices:
    # any number from 1 to 5 is accepted (results come back as floats, e.g. 5.0).
    # Compare with DiscreteMetric above, where a LIST means "pick one of these labels".
    allowed_values=(
        1,
        5,
    ),

)

In [38]:
# STEP 17: The experiment, i.e. what to do for ONE row of the dataset.
# Ragas calls this function once per row (several rows at a time) when we run
# run_rag_evaluation.arun(dataset) inside main(). For each row:
#   1. run the RAG app on the question
#   2. score the answer with both metrics (the judge LLM)
#   3. return a dict, which becomes one row of the results table
# Cost per row: 1 embedding call + 1 generator call + 2 judge calls.
@experiment()
async def run_rag_evaluation(row):

    # --------------------------------------------------------
    # Get question
    # --------------------------------------------------------
    question = row["question"]

    # --------------------------------------------------------
    # Get expected answer
    # --------------------------------------------------------
    expected_answer = row["expected_answer"]


    # ========================================================
    # RUN LANGCHAIN RAG
    # ========================================================

    # Run the system under test. The judge only sees the final answer, but we keep
    # the retrieval information too, for debugging.
    rag_result = await rag_query(question)

    response = rag_result["answer"]

    retrieved_contexts = rag_result["retrieved_contexts"]

    retrieved_document_ids = rag_result["retrieved_document_ids"]


    # ========================================================
    # RAGAS CORRECTNESS EVALUATION
    # ========================================================

    # The keyword arguments fill the metric prompt's placeholders
    # ({question}, {expected_answer}, {response}). Returns a result object with:
    #   .value  -> "pass" or "fail"
    #   .reason -> the judge's explanation
    # (Expert tip: the two metric calls are independent, so asyncio.gather(...) could
    # run them in parallel.)
    correctness_result = await correctness_metric.ascore(
        llm=evaluator_llm,
        question=question,
        expected_answer=expected_answer,
        response=response,
    )


    # ========================================================
    # RAGAS RELEVANCE EVALUATION
    # ========================================================

    # Reference-free: only the question and the response are needed.
    relevance_result = await relevance_metric.ascore(
        llm=evaluator_llm,
        question=question,
        response=response,
    )


    # ========================================================
    # RETURN EXPERIMENT DATA
    # ========================================================

    # Every key returned here becomes a column in the results table and CSV.
    # Always keep the reasons. When a score looks wrong, the reason shows whether the
    # app or the judge is at fault.
    return {
        # **row copies the original columns (question, expected_answer) into the result,
        # so each result row is self-contained.
        **row,

        # Application Output
        "response": response,

        # Retrieval Information
        "retrieved_contexts": retrieved_contexts,
        "retrieved_document_ids": retrieved_document_ids,

        # Correctness
        "correctness": correctness_result.value,
        "correctness_reason": correctness_result.reason,

        # Relevance
        "relevance": relevance_result.value,
        "relevance_reason": relevance_result.reason,
    }

In [39]:
# STEP 18: The orchestrator:
#   load data -> run the experiment -> summarise -> save -> print the judge's reasons.
# Defining main() does NOT run it. The next cell does.
async def main():

    print("\n" + "=" * 100)
    print("RAGAS + LANGCHAIN EVALUATION")
    print("=" * 100)

    # ========================================================
    # LOAD DATASET
    # ========================================================

    # Builds the 4-row golden dataset and writes evals/datasets/ragas_langchain_dataset.csv.
    dataset = load_dataset()

    print("\nDataset loaded.")

    # ========================================================
    # RUN RAGAS EXPERIMENT
    # ========================================================

    # Runs run_rag_evaluation on every row concurrently (that's the progress bar in
    # the output). Ragas also saves its own copy of the results as
    # evals/experiments/<name>.csv (here: ragas_langchain_experiment.csv).
    # Give each run a descriptive name (e.g. "gpt4o_topk3_loose_prompt") so you can
    # compare versions side by side. Reusing a name can overwrite earlier results.
    results = await run_rag_evaluation.arun(
        dataset,
        name="ragas_langchain_experiment",
    )

    # ========================================================
    # CONVERT RESULTS TO PANDAS
    # ========================================================

    # One row per question, one column per key returned by run_rag_evaluation.
    # Rows run concurrently, so the row order may differ from the dataset order.
    df = results.to_pandas()

    # ========================================================
    # DETAILED RESULTS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED RESULTS")
    print("=" * 100)

    # Show only the key columns. The full text and reasons are printed at the end.
    columns_to_show = [
        "question",
        "response",
        "correctness",
        "relevance",
    ]

    print(df[columns_to_show].to_string(index=False))

    # ========================================================
    # CORRECTNESS PASS RATE
    # ========================================================

    # Pass rate = the share of rows labelled "pass".
    #   .astype(str).str.lower() -> tolerate "PASS" / "Pass"
    #   .eq("pass")              -> True / False for each row
    #   .mean()                  -> mean of booleans (True=1, False=0) = fraction of passes
    correctness_pass_rate = (
        df["correctness"]
        .astype(str)
        .str.lower()
        .eq("pass")
        .mean()
    )

    # ========================================================
    # AVERAGE RELEVANCE
    # ========================================================

    # errors="coerce" turns anything non-numeric (e.g. a missing score) into NaN
    # instead of crashing, and .mean() skips NaN values.
    average_relevance = pd.to_numeric(
        df["relevance"],
        errors="coerce",
    ).mean()

    # ========================================================
    # SUMMARY
    # ========================================================

    print("\n" + "=" * 100)
    print("SUMMARY")
    print("=" * 100)

    print(f"Correctness Pass Rate : {correctness_pass_rate:.1%}")
    print(f"Average Relevance     : {average_relevance:.2f}/5")

    # ========================================================
    # SAVE RESULTS
    # ========================================================

    # Save our own complete copy (every column, including reasons and retrieved doc IDs).
    # parents=True also creates evals/ if it doesn't exist.
    experiment_directory = Path("evals/experiments")
    experiment_directory.mkdir(parents=True, exist_ok=True)

    output_file = experiment_directory / "ragas_langchain_results.csv"

    df.to_csv(output_file, index=False)

    print("\nResults saved to:")
    print(output_file)

    # ========================================================
    # PRINT DETAILED REASONS
    # ========================================================

    print("\n" + "=" * 100)
    print("DETAILED EVALUATION REASONS")
    print("=" * 100)

    # Print every judge reason next to its question and answer. This is often the most
    # useful part of an evaluation: the numbers tell you THAT something is wrong, and
    # the reasons (plus the retrieved doc IDs) tell you WHY.
    for _, result_row in df.iterrows():

        print("\nQUESTION:")
        print(result_row["question"])

        print("\nRESPONSE:")
        print(result_row["response"])

        print("\nCORRECTNESS:")
        print(result_row["correctness"])

        print("\nCORRECTNESS REASON:")
        print(result_row["correctness_reason"])

        print("\nRELEVANCE:")
        print(result_row["relevance"])

        print("\nRELEVANCE REASON:")
        print(result_row["relevance_reason"])

        print("\nRETRIEVED DOCUMENT IDS:")
        print(result_row["retrieved_document_ids"])

        print("\n" + "-" * 100)

In [40]:
# STEP 19: Run everything.
#
# asyncio.run() is the normal way to start async code in a .py script. Jupyter,
# however, already runs its own event loop, so asyncio.run() raises
#   RuntimeError: asyncio.run() cannot be called from a running event loop
# The except-block catches that, applies nest_asyncio (a patch that allows a loop
# inside the running loop), and runs main() again. That way the same cell works in
# a script AND in a notebook. Requires: pip install nest_asyncio
#
# Side effect: the first main() object, created for asyncio.run(), is never
# awaited. That's why the output shows a harmless
# "RuntimeWarning: coroutine 'main' was never awaited".
# Simpler notebook-only alternative: Jupyter supports top-level await, so a cell
# containing just `await main()` would also work.
try:
    asyncio.run(main())

except RuntimeError as error:

    # Jupyter Notebook fallback
    if "asyncio.run() cannot be called" in str(error) or "event loop" in str(error).lower():
        import nest_asyncio
        nest_asyncio.apply()

        loop = asyncio.get_event_loop()
        loop.run_until_complete(main())

    else:
        raise


RAGAS + LANGCHAIN EVALUATION

Dataset loaded.


Running experiment: 100%|██████████| 4/4 [00:06<00:00,  1.66s/it]


DETAILED RESULTS
                      question                                                                                                                                                                                           response correctness  relevance
                What is Ragas?                                                                                               Ragas is an open-source evaluation framework designed for evaluating and improving LLM applications.        pass        5.0
     What do Ragas metrics do? Ragas metrics measure different properties of AI systems, including factual correctness, faithfulness, answer relevancy, retrieval quality, agent behavior, and application-specific requirements.        pass        5.0
How do Ragas experiments work?                                                                                                                                                        I don't know based on the provided context.        fa


/var/folders/rk/q1h995hd56v2jbhjyxn_jzsm0000gn/T/ipykernel_88669/3836954189.py:12: RuntimeWarning: coroutine 'main' was never awaited
  loop.run_until_complete(main())


## What the results tell us (from the saved run)

These numbers come from the run whose output is saved above. LLM outputs vary a little between runs, so your numbers may differ.

| Question | Retrieved doc IDs | Correctness | Relevance | What we learn |
|---|---|---|---|---|
| What is Ragas? | `ragas_intro`, `rag_intro` | pass | 5 | The right document ranked first and the answer is grounded in it. |
| What do Ragas metrics do? | `ragas_metrics`, `ragas_intro` | pass | 5 | Same. |
| How do Ragas experiments work? | **`ragas_experiments`**, `ragas_intro` | **fail** | **1** | **Generation failure, not retrieval failure.** The correct document *was* retrieved (ranked #1), but the model answered *"I don't know based on the provided context."* The strict "ONLY use the context / say I don't know" prompt made it over-cautious: the context says what experiments *allow*, while the question asks how they *work*. |
| What is RAG? | `rag_intro`, `ragas_intro` | **fail** | **5** | **Relevant but incomplete.** The answer only expanded the acronym. The prompt's "keep the answer concise" rule dropped the explanation the expected answer requires. Relevance (on-topic?) and correctness (complete and right?) measure different things, which is why we use both. |

**Summary:** 2/4 pass (50%) and an average relevance of 3.5/5. This matches the `correctness_pass_rate` and `average_relevance` calculations in `main()`.

### Takeaways

1. **Log the evidence, not just the answer.** Without `retrieved_document_ids` you couldn't tell that the "experiments" failure was the prompt's fault rather than the retriever's.
2. **Read the judge's reasons.** The aggregate score tells you *that* something is wrong. The reasons tell you *why*.
3. **Prompt rules involve trade-offs.** The anti-hallucination rules ("only use context", "be concise") cost two correct answers here. Evaluation lets you measure that trade-off instead of guessing.

### Ideas to try next

* Loosen the prompt (for example, allow a 2-3 sentence answer, or tell the model to answer when the context *partially* covers the question). Re-run under a new experiment `name` and compare.
* Change `TOP_K` and watch how both retrieval and the scores change.
* Add questions whose answers are **not** in the knowledge base, to check that the app correctly says "I don't know".
* Add Ragas' built-in `Faithfulness` and `ContextPrecision`/`ContextRecall` metrics (see the import catalog at the top) using `retrieved_contexts`.
